# Lab: Intelligent Document Processing

**This is an OPTIONAL lab. We will NOT dedicate class time for this lab. We recommend you try this lab outside of the dedicated class time. If you happen to have free extra time leftover during the class, please try this out as you wish!**

📚 In this lab you will learn and practice the following:

❄️ Use AI_PARSE_DOCUMENT to extract text and structure from PDFs in LAYOUT and OCR modes

❄️ Extract embedded images from documents and analyze them with AI functions

❄️ Use AI_EXTRACT to extract structured data from images using a defined schema

❄️ Build automated document processing pipelines with stored procedures

❄️ Create dynamic tables with Cortex AI functions for automatic incremental processing

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any function? Ask CoCo *"What does [function name] do?"* to get its syntax, supported options, and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Introduction

Intelligent Document Processing (IDP) automates the extraction of valuable information from unstructured documents and images. This lab covers two complementary Snowflake Cortex AI functions:

- **AI_PARSE_DOCUMENT**: Extract text and layout from PDFs and documents stored on a Snowflake stage
- **AI_EXTRACT**: Extract structured fields from images and documents using a defined schema

We will work with TravelBug PDF documents and hiking equipment product images to demonstrate both functions, then build an automated extraction pipeline using stored procedures and dynamic tables.

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
# Set up session context and display current settings
from snowflake.snowpark.context import get_active_session

session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Set_up_your_current_context_for_the_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Intelligent Document Processing';
SHOW PARAMETERS LIKE 'query_tag' in session 
  ->> SELECT "value" AS query_tag FROM $1;

## AI_PARSE_DOCUMENT

Extracts text, tables, and layout information from PDFs and Word documents stored on a Snowflake stage. Two modes:
- **LAYOUT**: returns markdown-formatted output preserving tables and document structure
- **OCR**: returns plain text content only

### AI_PARSE_DOCUMENT use cases.

| Industry | Use Case | Business Value |
| :--- | :--- | :--- |
| **Financial Services & Accounting** 🧾 | **Automated Invoice Processing** | **Reduces operational costs** and accelerates payment cycles by eliminating manual data entry and minimizing errors. |
| **Banking & Lending** 🏦 | **Loan Application Automation** | **Improves customer experience** and increases loan processing capacity by providing faster, more accurate approval decisions. |
| **Insurance** 📄 | **Claims Form Digitization** | **Speeds up claims resolution** and reduces administrative overhead, leading to higher customer satisfaction and faster payouts. |

### Extract text from a PDF document.

Run AI_PARSE_DOCUMENT in LAYOUT mode against a balloon review PDF and observe how the output preserves structure as markdown.

In [ ]:
%%sql -r Extract_Balloon_Review_sql
-- Extract text from a balloon review PDF using LAYOUT mode
SELECT AI_PARSE_DOCUMENT (
    TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files','balloon_review3.pdf'),
    {'mode': 'LAYOUT' , 'page_split': true}) AS layout;

In [ ]:
# Format output from the previous cell
import json

df = Extract_Balloon_Review_sql.to_pandas()

# Get the LAYOUT column value
generated_response = df["LAYOUT"].iloc[0]

# Parse the JSON response and extract the page content
parsed = json.loads(generated_response)
content = parsed['pages'][0]['content']

# Replace the literal '\\n' string with a true newline character '\n'
corrected_text = content.replace('\\n', '\n')

print(corrected_text)

📌 **OCR vs LAYOUT**:
- **LAYOUT** mode returns markdown with tables and structural elements preserved - use this when document structure matters (e.g., for RAG pipelines or targeted extraction).
- **OCR** mode returns plain text only - use this for simple text extraction from text-heavy documents.

In [ ]:
%%sql -r OCR_Mode_Demo_sql
SELECT
  SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
    TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files', 'travelbug_checklist.pdf'),
    {'mode': 'OCR'}
  )::STRING AS document_text;

In [ ]:
# Format output from the previous cell
import json

df = OCR_Mode_Demo_sql.to_pandas()

# Get the JSON string from the DataFrame
generated_response = df["DOCUMENT_TEXT"].iloc[0]

# Parse the JSON and extract the content
parsed = json.loads(generated_response)
content_text = parsed["content"]

# Display with markdown (newlines are already real newlines)
print(content_text)

### Extract structured data from a document with table layout.

Parse the TravelBug activity checklist PDF in LAYOUT mode, store the result in a table, then use SPLIT_PART to extract the structured columns from the markdown output.

In [ ]:
%%sql -r Create_Travel_Checklist_sql
-- Parse the TravelBug checklist PDF and store the result
CREATE OR REPLACE TABLE {{user}}_genai_db.raw.travel_checklist AS
SELECT AI_PARSE_DOCUMENT (
    TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files','travelbug_checklist.pdf') ,
    {'mode': 'LAYOUT' , 'page_split': false}  ) AS layout;

In [ ]:
%%sql -r Split_Part_Demo_sql
-- Select the three columns for the final table.
SELECT
    -- Split each line by '|' and take the 2nd part for the "Activity".
    -- TRIM() removes any leading/trailing spaces.
    TRIM(SPLIT_PART(s.value, '|', 2)) AS "Activity",

    -- Take the 3rd part for "What to bring". The REPLACE function also
    -- removes the HTML <br> tag used for line breaks within a cell.
    REPLACE(TRIM(SPLIT_PART(s.value, '|', 3)), '<br>', '') AS "What to bring",

    -- Take the 4th part for "Provided to you".
    TRIM(SPLIT_PART(s.value, '|', 4)) AS "Provided to you"
FROM
    -- Your source table.
    {{user}}_genai_db.raw.travel_checklist,

    -- This function splits the single 'content' text block into multiple virtual rows.
    -- The ::VARCHAR explicitly converts the VARIANT data to a string before splitting.
    LATERAL SPLIT_TO_TABLE(layout:content::VARCHAR, '\n') AS s
WHERE
    -- Filter for lines that are part of the markdown table data.
    s.value LIKE '| %'

    -- Exclude the markdown table's alignment/separator line.
    AND s.value NOT LIKE '| :--%'
    
    -- ADDED: Exclude the original header row itself to prevent it from appearing as data.
    AND s.value NOT LIKE '| Activity %';

### Extract embedded images from a document.

Use `extract_images: true` with LAYOUT mode to pull images embedded in a PDF. The output includes base64-encoded image data that can be displayed inline or passed to other AI functions.

In [ ]:
%%sql -r Extract_Images_sql
-- Extract embedded images from a PDF document
SELECT 
    AI_PARSE_DOCUMENT(
        TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files', 'travelbug_brochure.pdf'),
        {'mode': 'LAYOUT', 'extract_images': true}
    ):images AS extracted_images,
    ARRAY_SIZE(
        AI_PARSE_DOCUMENT(
            TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files', 'travelbug_brochure.pdf'),
            {'mode': 'LAYOUT', 'extract_images': true}
        ):images
    ) AS image_count

In [ ]:
# Format output from the previous cell
import json
import base64
from IPython.display import display, Image

df = Extract_Images_sql.to_pandas()

# Get the JSON string from the first row
json_string = df.iloc[0, 0]

# Parse and display each image
data = json.loads(json_string)

for item in data:
    print(f"\n--- {item['id']} ---")
    img_b64 = item['image_base64']
    # Strip data URI prefix if present (e.g., 'data:image/jpeg;base64,')
    if ',' in img_b64 and img_b64.startswith('data:'):
        img_b64 = img_b64.split(',', 1)[1]
    img_bytes = base64.b64decode(img_b64)
    display(Image(data=img_bytes, width=400))

### Describe extracted images using AI_EXTRACT.

Pass the base64-encoded image data from AI_PARSE_DOCUMENT directly into AI_EXTRACT to generate a description - this is the multimodal pipeline: PDF → image extraction → AI analysis.

In [ ]:
%%sql -r Describe_Images_sql
SELECT AI_EXTRACT(
    file_data => BASE64_DECODE_BINARY(
        REGEXP_REPLACE(
            (SELECT (
                AI_PARSE_DOCUMENT(
                    TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files', 'travelbug_brochure.pdf'),
                    {'mode': 'LAYOUT', 'extract_images': true}
                ):images[0]['image_base64']
            )::STRING),
            '^data:image/[^;]+;base64,', ''
        )
    ),
    responseFormat => {'description': 'Describe what is shown in this image'}
) AS image_description

In [ ]:
# Format output from the previous cell
import json

df = Describe_Images_sql.to_pandas()

# Get the JSON string from the DataFrame
generated_response = df["IMAGE_DESCRIPTION"].iloc[0]

# Parse JSON and extract description
data = json.loads(generated_response)
description = data['response']['description']

# Display
print(description)

## Explore Source Files

First, let's explore the hiking equipment images available in our stage. These images contain product information that we'll extract using AI.

In [ ]:
%%sql -r Explore_Source_Files_list_sql
LIST @{{user}}_GENAI_DB.RESOURCES.GENAI2DAY/source_files/

In [ ]:
%%sql -r Explore_Source_Files_sql
-- List hiking equipment image files on stage
LIST @{{user}}_GENAI_DB.RESOURCES.GENAI2DAY/source_files/ PATTERN='.*\.png';

In [ ]:
%%sql -r Explore_Source_Files_list_images_sql
-- List hiking equipment images with metadata from directory table
SELECT 
    RELATIVE_PATH,
    SIZE,
    LAST_MODIFIED
FROM DIRECTORY('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY')
WHERE RELATIVE_PATH LIKE 'source_files/travelbug_rental%'
  AND (RELATIVE_PATH LIKE '%.png' OR RELATIVE_PATH LIKE '%.jpg')

## Extract Data with AI_EXTRACT

`AI_EXTRACT` identifies and extracts structured fields from images and documents using a JSON schema you define. Works with images (PNG, JPG), PDFs, and text files.

The `AI_EXTRACT` function uses a vision-based large language model to extract structured information from images and documents. We'll define a schema to extract:
- **ID**: Product identifier
- **Description**: Product name/description
- **Price**: Product price

Let's test it on a single image first.

In [ ]:
%%sql -r Extract_Data_with_AI_EXTRACT_sql
-- Extract data from a single hiking equipment image
SELECT 
    'source_files/travelbug_rental_backpack_40l.png' AS file_path,
    AI_EXTRACT(
        file => TO_FILE('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY', 'source_files/travelbug_rental_backpack_40l.png'),
        responseFormat => PARSE_JSON('{
            "schema": {
                "type": "object",
                "properties": {
                    "id": {"description": "Product ID or item number", "type": "string"},
                    "description": {"description": "Description of the hiking equipment", "type": "string"},
                    "price": {"description": "Price of the item", "type": "string"}
                }
            }
        }')
    ) AS extracted_data

In [ ]:
# Format output from the previous cell
import json

df = Extract_Data_with_AI_EXTRACT_sql
rows = df.collect()

for row in rows:
    file_path = row[0]
    data = json.loads(row[1])
    response = data['response']
    
    print(f"**File:** {file_path}")
    print(f"- **Description:** {response['description']}")
    print(f"- **ID:** {response['id']}")
    print(f"- **Price:** ${response['price']}")
    print("---")

## Process All Images

Now let's extract data from all hiking equipment images in our stage. We'll use the DIRECTORY table function to iterate through files.

In [ ]:
%%sql -r Process_All_Images_sql
-- Extract data from all hiking equipment images
SELECT 
    RELATIVE_PATH AS file_path,
    AI_EXTRACT(
        file => TO_FILE('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY', RELATIVE_PATH),
        responseFormat => PARSE_JSON('{
            "schema": {
                "type": "object",
                "properties": {
                    "id": {"description": "Product ID or item number", "type": "string"},
                    "description": {"description": "Description of the hiking equipment", "type": "string"},
                    "price": {"description": "Price of the item", "type": "string"}
                }
            }
        }')
    ) AS extracted_data
FROM DIRECTORY('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY')
WHERE RELATIVE_PATH LIKE 'source_files/travelbug_rental%'
  AND (RELATIVE_PATH LIKE '%.png' OR RELATIVE_PATH LIKE '%.jpg')

In [ ]:
# Format output from the previous cell
import json

df = Process_All_Images_sql.to_pandas()

for _, row in df.iterrows():
    file_path = row.iloc[0]
    data = json.loads(row.iloc[1])
    response = data['response']
    
    print(f"**File:** {file_path}")
    print(f"- **Description:** {response['description']}")
    print(f"- **ID:** {response['id']}")
    print(f"- **Price:** ${response['price']}")
    print("---")

## Create Target Table

Let's create a table to store our extracted data in a structured format.

In [ ]:
%%sql -r Create_Target_Table_sql
-- Create table to store extracted hiking equipment data
CREATE TABLE IF NOT EXISTS {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_EXTRACTED (
    file_path VARCHAR,
    extracted_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    id VARCHAR,
    description VARCHAR,
    price VARCHAR,
    raw_response VARIANT
)

## Build the Extraction Pipeline

Now we'll create a stored procedure that extracts data from images and inserts it into our target table. This procedure will skip files that have already been processed.

In [ ]:
%%sql -r Build_the_Extraction_Pipeline_sql
-- Create stored procedure for document extraction pipeline
CREATE OR REPLACE PROCEDURE {{user}}_GENAI_DB.RAW.EXTRACT_HIKING_EQUIPMENT()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    INSERT INTO {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_EXTRACTED 
        (file_path, extracted_at, id, description, price, raw_response)
    SELECT 
        RELATIVE_PATH AS file_path,
        CURRENT_TIMESTAMP() AS extracted_at,
        response:response:id::VARCHAR AS id,
        response:response:description::VARCHAR AS description,
        response:response:price::VARCHAR AS price,
        response AS raw_response
    FROM (
        SELECT 
            RELATIVE_PATH,
            AI_EXTRACT(
                file => TO_FILE('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY', RELATIVE_PATH),
                responseFormat => PARSE_JSON('{
                    "schema": {
                        "type": "object",
                        "properties": {
                            "id": {"description": "Product ID or item number", "type": "string"},
                            "description": {"description": "Description of the hiking equipment", "type": "string"},
                            "price": {"description": "Price of the item", "type": "string"}
                        }
                    }
                }')
            ) AS response
        FROM DIRECTORY('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY')
        WHERE RELATIVE_PATH LIKE 'source_files/travelbug_rental%'
          AND (RELATIVE_PATH LIKE '%.png' OR RELATIVE_PATH LIKE '%.jpg')
          AND RELATIVE_PATH NOT IN (SELECT file_path FROM {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_EXTRACTED)
    );
    
    RETURN 'Extraction completed successfully';
END;
$$

In [ ]:
%%sql -r Build_Extraction_Pipeline_run_sql
-- Run the extraction pipeline
CALL {{user}}_GENAI_DB.RAW.EXTRACT_HIKING_EQUIPMENT()

In [ ]:
%%sql -r Build_Extraction_Pipeline_view_results_sql
-- View extracted hiking equipment data
SELECT 
    file_path,
    id,
    description,
    price,
    extracted_at
FROM {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_EXTRACTED
ORDER BY description

## Dynamic Tables with Cortex AI

Dynamic tables can automatically process new documents as they arrive using incremental refresh with Cortex AI functions. This provides a declarative alternative to the stored procedure approach.

**Key Benefits:**
- **No stored procedure needed** - Dynamic table handles the pipeline automatically
- **Incremental refresh** - Only processes NEW rows added to the source table
- **AI enrichment** - Can combine multiple AI functions (e.g., AI_EXTRACT + AI_CLASSIFY)

> **Note:** Dynamic tables don't support sources that include directory tables, external tables, streams, and materialized views ([documentation](https://docs.snowflake.com/en/user-guide/dynamic-tables-limitations)). Therefore, we first create a base table from the directory listing.

In [ ]:
%%sql -r Dynamic_Tables_create_source_table_sql
-- Create source table from directory listing (required for dynamic tables)
CREATE OR REPLACE TABLE {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_FILES AS
SELECT RELATIVE_PATH AS file_path
FROM DIRECTORY('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY')
WHERE RELATIVE_PATH LIKE 'source_files/travelbug_rental%'
  AND (RELATIVE_PATH LIKE '%.png' OR RELATIVE_PATH LIKE '%.jpg')

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE DYNAMIC TABLE {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_DT
    TARGET_LAG = '1 hour'
    WAREHOUSE = {{user}}_genai_wh
    AS
    SELECT 
        file_path,
        response:response:"What is the product ID?"::VARCHAR AS id,
        response:response:"What is the product description?"::VARCHAR AS description,
        response:response:"What is the price?"::VARCHAR AS price,
        AI_CLASSIFY(
            response:response:"What is the product description?"::VARCHAR,
            ['Footwear', 'Backpacks', 'Camping Gear', 'Accessories', 'Clothing']
        ):labels[0]::VARCHAR AS category,
        response AS raw_response
    FROM (
        SELECT 
            file_path,
            AI_EXTRACT(
                file => TO_FILE('@{{user}}_GENAI_DB.RESOURCES.GENAI2DAY', file_path),
                responseFormat => ['What is the product ID?', 'What is the product description?', 'What is the price?']
            ) AS response
        FROM {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_FILES
    );

In [ ]:
%%sql -r Dynamic_Tables_view_results_sql
-- View the dynamic table results with AI-generated categories
SELECT file_path, id, description, price, category 
FROM {{user}}_GENAI_DB.RAW.HIKING_EQUIPMENT_DT

## View Extracted Equipment Catalog

Now that the dynamic table has processed our hiking equipment images, let's visualize the catalog. The cell below displays product images alongside their extracted details and AI-generated categories, making it easy to browse and verify the extraction results.

In [ ]:
# Display extracted equipment catalog with images from the dynamic table
import base64
import requests
from snowflake.snowpark.context import get_active_session
from IPython.display import display, Image, HTML

print("=== Hiking Equipment Dashboard ===")

session = get_active_session()
user = session.get_current_user().replace('"', '')

df = session.sql(f"""
    SELECT 
        id, 
        description, 
        price,
        category,
        file_path,
        GET_PRESIGNED_URL(@{user}_GENAI_DB.RESOURCES.GENAI2DAY, file_path, 3600) AS image_url
    FROM {user}_GENAI_DB.RAW.HIKING_EQUIPMENT_DT
    ORDER BY category, description
""").to_pandas()

print(f"Total Items: {len(df)}\n")

for i, row in df.iterrows():
    print(f"Description: {row['DESCRIPTION']}")
    print(f"ID: {row['ID']}")
    print(f"Price: ${row['PRICE']}")
    print(f"Category: {row['CATEGORY']}")
    try:
        resp = requests.get(row['IMAGE_URL'])
        if resp.status_code == 200:
            display(Image(data=resp.content, width=300))
    except Exception as e:
        print(f"  (Could not load image: {e})")
    print("---")

## Arctic-Extract Fine-Tuning

While the base **AI_EXTRACT** function works well for general documents, you can fine-tune the `arctic-extract` model to achieve higher accuracy for domain-specific documents such as TravelBug invoices, equipment specifications, or custom forms. Model fine-tuning is covered in a later module.

### When to Consider Fine-Tuning

| Scenario | Recommendation |
|:---|:---|
| General document extraction | Use base AI_EXTRACT |
| Domain-specific formats (invoices, forms) | Consider fine-tuning |
| Consistent extraction errors on specific fields | Fine-tuning recommended |
| High-volume production workloads | Fine-tuning for accuracy |

### How It Works

Fine-tuning `arctic-extract` uses Snowflake's **FINETUNE** function with a training Dataset containing:

| Column | Description | Example |
|:---|:---|:---|
| **File** | Path to document | `@db.schema.stage/file.pdf` |
| **Prompt** | Extraction questions/schema | `{'price': 'What is the price?'}` |
| **Response** | Expected answers (ground truth) | `{'price': '29.99'}` |

### Usage Notes

| Guideline | Details |
|:---|:---|
| Recommended minimum documents | At least 20 documents |
| Max pages per document | 64 (Oregon, Frankfurt) or 125 (N. Virginia, Azure East US 2) |
| Max unique document files | 1,000 (same file can be referenced multiple times) |
| Dataset size limit | questions × total pages ≤ 50,000 |

Once fine-tuned, you use your custom model by specifying the `model` parameter in AI_EXTRACT:

```sql
SELECT AI_EXTRACT(
    model => 'db.schema.my_tuned_extract_model',
    file => TO_FILE('@stage', 'document.png')
);
```

> **Note:** See the [documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/arctic-extract-finetuning) for details.

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from IPython.display import display, HTML

quiz_data = [
    {"q": "What function is used to extract structured data from images and documents in Snowflake?", "options": ["AI_COMPLETE", "AI_EXTRACT", "PARSE_DOCUMENT", "EXTRACT_TEXT"], "correct": "AI_EXTRACT"},
    {"q": "What is the purpose of defining an extraction schema when using AI_EXTRACT?", "options": ["To specify the file format", "To define exactly what fields to extract from documents", "To set the output table name", "To configure the AI model"], "correct": "To define exactly what fields to extract from documents"},
    {"q": "Which SQL object can be used to automate document extraction for multiple files?", "options": ["View", "Stored Procedure", "Stream", "Sequence"], "correct": "Stored Procedure"},
    {"q": "What is the difference between LAYOUT and OCR modes in AI_PARSE_DOCUMENT?", "options": ["LAYOUT extracts images while OCR extracts text", "LAYOUT preserves document structure as markdown while OCR returns plain text only", "LAYOUT is faster while OCR is more accurate", "LAYOUT works on PDFs while OCR works on images"], "correct": "LAYOUT preserves document structure as markdown while OCR returns plain text only"},
    {"q": "What advantage do dynamic tables with Cortex AI provide over stored procedures for document processing?", "options": ["They run faster than stored procedures", "They support more file formats", "They automatically process new data incrementally without scheduling", "They can extract more fields per document"], "correct": "They automatically process new data incrementally without scheduling"},
]

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for oi, opt in enumerate(item["options"]):
        is_correct = opt == item["correct"]
        css_class = 'ok' if is_correct else 'no'
        feedback = '\u2705 Correct!' if is_correct else '\u274c Try again'
        uid = f'cq{qi}_opt{oi}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ **AI_PARSE_DOCUMENT** extracts text, tables, and layout from PDFs in two modes: LAYOUT (preserves structure as markdown) and OCR (plain text extraction).

❄️ **AI_EXTRACT** extracts structured fields from images and documents using a JSON schema you define, with no manual data entry needed.

❄️ **Multimodal pipelines** combine both functions: parse a document for embedded images, then extract structured data from those images using AI.

❄️ **Stored procedures** automate extraction pipelines, processing multiple documents in a single operation with duplicate-handling built in.

❄️ **Dynamic tables with Cortex AI** provide declarative, incremental pipelines that automatically process new data as it arrives without scheduling or stored procedures required.

❄️ **Fine-tuning arctic-extract** can improve accuracy for domain-specific documents (invoices, forms) when base model extraction isn't sufficient.

❄️ Extracted data stored in Snowflake tables integrates seamlessly with downstream analytics, dashboards, and reporting.